#### Ollama Embeddings

In [33]:
from langchain_ollama import OllamaEmbeddings

In [34]:
embeddings = OllamaEmbeddings(base_url="http://localhost:11434", model="nomic-embed-text")

#### Vector Store

In [35]:
from pymongo import MongoClient
from langchain_mongodb import MongoDBAtlasVectorSearch

In [36]:
CONNECTION_STRING = "mongodb://127.0.0.1:27017/?directConnection=true&serverSelectionTimeoutMS=2000&appName=mongosh+2.5.10"
DB_NAME = "ratingsdb"
COLLECTION_NAME = "ratingsEmbeddings"

# MongoClient
client = MongoClient(CONNECTION_STRING)

vector_store = MongoDBAtlasVectorSearch(
    collection=client[DB_NAME][COLLECTION_NAME],
    embedding=embeddings,
    text_key="message",
    index_name="vector_index",
    embedding_key="ratings_embedding"
)

# Use the vector store as retriever
retriever = vector_store.as_retriever(search_type="similarity",
                                      search_kwargs={"k": 5, "pre_filter": {"stars": {"$ne": None}}})

#### LLM

In [37]:
from langchain_ollama import OllamaLLM
llm = OllamaLLM(model="smollm2:135m", base_url="http://localhost:11434",  temperature=0)

#### RAG Chain

In [38]:

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant.
Use ONLY the following context to answer the question.

Context:
{context}

Question:
{question}

Answer:
"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [39]:
# answer = rag_chain.invoke("Is the review from user_id 10 positive or negative?")

docs = retriever.invoke("Is there any comment about peanuts?")

for doc in docs:
    print(doc.page_content)



more peanuts please
more peanuts please
more peanuts please
more peanuts please
more peanuts please


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

def query_rag(query):

    retriever = vector_store.as_retriever(search_type="similarity",
                                      search_kwargs={"k": 5, "pre_filter": {"stars": {"$ne": None}}})
    

    template = """
    Use the following pieces of context to answer the question at the end.
    If you don't know the answer, just say that you don't know, don't try to make up an answer.
    Do not answer the question if there is no given context.
    Do not answer the question if it is not related to the context.
    Context: {context}
    Question: {question}
    """

    custom_rag_prompt = ChatPromptTemplate.from_template(template)

    retrieve = {
        "context": retriever | (lambda docs: "\n\n".join([d.page_content for d in docs])), 
        "question": RunnablePassthrough()
        }
    
    llm = llm = OllamaLLM(model="smollm2:135m", base_url="http://localhost:11434",  temperature=0)

    response_parser = StrOutputParser()

    rag_chain = (
        retrieve
        | custom_rag_prompt
        | llm
        | response_parser
    )

    answer = rag_chain.invoke(query)
    return answer


In [46]:
query_rag("What are comments about airport")

'Comments about airport: The comment is that the airport refurb looks great.'

In [47]:
query_rag("What are comments on peanuts?")

'Comments about peanuts include:\n\n1. "More peanuts please" - This is a common phrase used to encourage customers to buy more peanuts, often in an attempt to boost sales or attract new customers.\n\n2. "Do you need more peanuts?" - This question asks for the quantity of peanuts that would be needed to satisfy customer demand, rather than asking about their quality or availability.\n\n3. "More peanuts please" - This phrase is used to encourage customers to buy more peanuts by implying that they are in a hurry and want something extra.\n\nIn summary, comments on peanuts typically include requests for quantity (more), quality (more), or urgency (buying more).'